# ДЗ 3. NumPy на данных «Дала Маркет»
Срок: **до четверга 12.11, 19:00**.
Сдача — папка `hw03` репозитория `dsa-homework` (`hw03.ipynb`): `git add hw03` → `git commit` → `git push`.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем — только массивы и маски (кроме задачи 8).

- **[База]** обязательно для всех · **[Стандарт]** — сколько получится · **[Челлендж]** — для тех, кому легко.

## [Для всех] Чтение кода — без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
# 1
a = np.arange(6).reshape(2, 3); print(a[1, 2], a.shape)
# 2
a = np.array([3, 8, 1, 8]); print(a.argmax(), a.max())
# 3
a = np.arange(10); b = a[::3]; b[0] = 99; print(a[0])
# 4
a = np.array([1, 5, 7, 2]); print((a > 3).sum(), (a > 3).mean())
# 5
print(np.zeros((2, 3)).shape, np.linspace(0, 1, 5)[1])
```

Мои предсказания и объяснения:

1. `5 (2, 3)` — индексирование начинается с нуля; во второй строке и третьем столбце стоит 5.
2. `1 8` — `argmax()` возвращает индекс первого максимума, `max()` — его значение.
3. `99` — `a[::3]` является представлением массива, поэтому изменение `b` меняет `a`.
4. `2 0.5` — условию удовлетворяют два элемента из четырёх; сумма маски даёт количество, среднее — долю.
5. `(2, 3) 0.25` — массив имеет две строки и три столбца; `linspace` включает обе границы.


In [ ]:
import numpy as np
a = np.arange(6).reshape(2, 3); print(a[1, 2], a.shape)
a = np.array([3, 8, 1, 8]); print(a.argmax(), a.max())
a = np.arange(10); b = a[::3]; b[0] = 99; print(a[0])
a = np.array([1, 5, 7, 2]); print((a > 3).sum(), (a > 3).mean())
print(np.zeros((2, 3)).shape, np.linspace(0, 1, 5)[1])

5 (2, 3)
1 8
99
2 0.5
(2, 3) 0.25


In [ ]:
from pathlib import Path
# orders.csv должен находиться рядом с ноутбуком. Поддерживаем и исходное имя загрузки.
data_path = Path('orders.csv')
if not data_path.exists():
    data_path = Path('orders (1).csv')
if not data_path.exists():
    try:
        from google.colab import files
        uploaded = files.upload()
        candidates = [name for name in uploaded if name.lower().endswith('.csv')]
        if len(candidates) != 1:
            raise ValueError('Загрузите один CSV-файл с заказами.')
        data_path = Path(candidates[0])
    except ImportError as exc:
        raise FileNotFoundError('Положите orders.csv рядом с ноутбуком.') from exc
print('Файл данных:', data_path)

Файл данных: orders.csv


In [ ]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open(data_path, encoding="utf-8-sig") as f:
    rows = list(csv.DictReader(f))

# функции из урока 2 + превращаем колонки в массивы NumPy
amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])

print(amount.shape, amount.dtype, paid.dtype)

(5463,) float64 bool


## [База]
1. Найдите позицию (`argmax`) самого дорогого заказа. Какая сумма, клиент и город?
2. Выручка оплаченных заказов и сумма неоплаченных — через маску `paid` и `~paid`.
3. Сколько заказов в Шымкенте и какая у них доля от всех заказов?

In [ ]:
# 1. Позиция максимума — индекс массива, а не order_id.
max_pos = amount.argmax()
print(f'1. Индекс {max_pos}, сумма {amount[max_pos]:,.0f} ₸, клиент {cust[max_pos]}, город {city[max_pos]}.')
print('Это тестовый заказ; argmax возвращает первый из одинаковых максимумов.')
print('Все позиции с максимальной суммой:', np.flatnonzero(amount == amount.max()))

# 2. Маска оплаты и её отрицание.
paid_revenue = amount[paid].sum()
unpaid_amount = amount[~paid].sum()
print(f'2. Оплачено {paid_revenue:,.0f} ₸, неоплачено {unpaid_amount:,.0f} ₸.')
print('Неоплаченная сумма относится к отменённым заказам и не является полученной выручкой.')

# 3. Город уже нормализован при чтении.
shymkent = city == 'Шымкент'
print(f'3. Шымкент: {shymkent.sum()} заказов, {shymkent.mean():.2%} от всех.')
print('Примерно каждый седьмой заказ относится к Шымкенту.')

1. Индекс 1001, сумма 9,999,990 ₸, клиент TEST, город Алматы.
Это тестовый заказ; argmax возвращает первый из одинаковых максимумов.
Все позиции с максимальной суммой: [1001 2268 2966]
2. Оплачено 101,909,360 ₸, неоплачено 7,974,000 ₸.
Неоплаченная сумма относится к отменённым заказам и не является полученной выручкой.
3. Шымкент: 804 заказов, 14.72% от всех.
Примерно каждый седьмой заказ относится к Шымкенту.


## [Стандарт]
4. Маркетинг предлагает скидку 15% на оплаченные заказы дороже 50 000 ₸. Сколько выручки магазин недополучит?
   Посчитайте дважды: со всеми заказами и без заказов клиента TEST. Почему ответы различаются в несколько раз?
5. Для **каждого** города — доля заказов, доставленных за 1–2 дня. Список городов: `np.unique(city)`; цикл по 6 городам здесь можно.
6. Ниже код, который AI написал на запрос «посчитай средний чек». Найдите две ошибки, исправьте и посчитайте правильно:
```python
avg = amount.mean()
print("Средний чек:", round(avg))
```

In [ ]:
# 4. Скидка распространяется только на оплаченные заказы строго дороже 50 000.
discount_mask = paid & (amount > 50_000)
real_customer = cust != 'TEST'
loss_all = (amount[discount_mask] * 0.15).sum()
loss_real = (amount[discount_mask & real_customer] * 0.15).sum()
print(f'4. Потери со всеми заказами: {loss_all:,.2f} ₸; без TEST: {loss_real:,.2f} ₸.')
print(f'Разница в {loss_all / loss_real:.2f} раза: три оплаченных заказа TEST по 9 999 990 ₸ завышают расчёт.')

# 5. Знаменатель — заказы города с известным сроком доставки.
known_days = ~np.isnan(days)
print('\n5. Быстрая доставка по городам:')
for name in np.unique(city):  # цикл по городам явно разрешён условием
    in_city = city == name
    eligible = in_city & known_days
    fast_count = (eligible & (days >= 1) & (days <= 2)).sum()
    total_known = eligible.sum()
    share = fast_count / total_known if total_known else np.nan
    print(f'{name}: {fast_count}/{total_known} = {share:.2%}; без срока: {(in_city & ~known_days).sum()}.')
print('Доля показывает скорость среди заказов с известным сроком; пропуски не считаем медленной доставкой.')

# 6. Средний чек: оплаченные заказы реальных клиентов.
# Две ошибки: не отобраны оплаченные заказы; не исключены тестовые заказы.
average_mask = paid & real_customer
avg = amount[average_mask].mean()
print(f'\n6. Средний чек без TEST по оплаченным заказам: {avg:,.2f} ₸ ({average_mask.sum()} заказ).')
print(f'Среднее по исходному ошибочному коду: {amount.mean():,.2f} ₸; оно включает отмены и TEST.')
# В основной части ДЗ строки не удаляем: условие задаёт исходные массивы.
# Дополнительный контроль качества: в материалах предыдущего урока отмечены полные дубли.
raw_table = np.genfromtxt(data_path, delimiter=',', dtype=str, skip_header=1, encoding='utf-8-sig')
_, first_indices = np.unique(raw_table, axis=0, return_index=True)
unique_rows = np.zeros(amount.size, dtype=bool)
unique_rows[first_indices] = True
avg_deduplicated = amount[average_mask & unique_rows].mean()
print(f'Дополнительно: полных дублей {(~unique_rows).sum()}; после их удаления средний чек {avg_deduplicated:,.2f} ₸.')
print('Этот дополнительный показатель использует другую выборку; основные ответы выше посчитаны на исходных строках.')

4. Потери со всеми заказами: 5,702,422.50 ₸; без TEST: 1,202,427.00 ₸.
Разница в 4.74 раза: три оплаченных заказа TEST по 9 999 990 ₸ завышают расчёт.

5. Быстрая доставка по городам:
Актобе: 247/393 = 62.85%; без срока: 4.
Алматы: 1412/2123 = 66.51%; без срока: 65.
Астана: 883/1350 = 65.41%; без срока: 40.
Атырау: 94/141 = 66.67%; без срока: 5.
Караганда: 365/514 = 71.01%; без срока: 24.
Шымкент: 240/782 = 30.69%; без срока: 22.
Доля показывает скорость среди заказов с известным сроком; пропуски не считаем медленной доставкой.

6. Средний чек без TEST по оплаченным заказам: 14,672.39 ₸ (4901 заказ).
Среднее по исходному ошибочному коду: 20,114.11 ₸; оно включает отмены и TEST.
Дополнительно: полных дублей 180; после их удаления средний чек 14,699.92 ₸.
Этот дополнительный показатель использует другую выборку; основные ответы выше посчитаны на исходных строках.


## [Челлендж]
7. Сравните скорость: сумма оплаченных заказов циклом `for` и через маску. Во сколько раз быстрее?
8. Смоделируйте A/B-тест. `np.random.seed(42)`. Группа A: 1 000 клиентов, вероятность покупки 5%; группа B: 1 000 клиентов, 6%
   (покупка = `np.random.rand(1000) < 0.05`). Сколько покупок в каждой группе?
   Повторите эксперимент 100 раз: в скольких случаях B оказалась лучше A? Что это говорит о размере выборки? (2–3 предложения)

In [ ]:
# 7. Цикл необходим именно для требуемого сравнения с NumPy.
# Это локальное исключение из общего запрета циклов в аналитических задачах.
from timeit import repeat

def sum_paid_loop():
    total = 0.0
    for value, is_paid in zip(amount, paid):
        if is_paid:
            total += value
    return total

def sum_paid_numpy():
    return amount[paid].sum()

assert np.isclose(sum_paid_loop(), sum_paid_numpy())
number = 200
loop_seconds = np.median(repeat(sum_paid_loop, number=number, repeat=5)) / number
numpy_seconds = np.median(repeat(sum_paid_numpy, number=number, repeat=5)) / number
print(f'7. Одинаковая сумма: {sum_paid_numpy():,.0f} ₸.')
print(f'Цикл: {loop_seconds * 1e6:.2f} мкс; NumPy: {numpy_seconds * 1e6:.2f} мкс; ускорение {loop_seconds / numpy_seconds:.2f} раза.')
print('Ускорение измерено на этом компьютере; на другом устройстве результат изменится.')

# 8. Один эксперимент в точности по условию.
np.random.seed(42)
buy_a = np.random.rand(1000) < 0.05
buy_b = np.random.rand(1000) < 0.06
print(f'\n8. Один эксперимент: A — {buy_a.sum()} покупок ({buy_a.mean():.1%}); B — {buy_b.sum()} ({buy_b.mean():.1%}).')

# Перезапускаем seed: первый из 100 опытов совпадает с одиночным выше.
np.random.seed(42)
results = np.empty((100, 2), dtype=int)
for trial in range(100):
    results[trial, 0] = (np.random.rand(1000) < 0.05).sum()
    results[trial, 1] = (np.random.rand(1000) < 0.06).sum()
b_wins = (results[:, 1] > results[:, 0]).sum()
a_wins = (results[:, 0] > results[:, 1]).sum()
ties = (results[:, 0] == results[:, 1]).sum()
print(f'100 опытов: B лучше A в {b_wins} случаях, A лучше B в {a_wins}, ничья в {ties}.')
print('При 1000 клиентах на группу разница в 1 процентный пункт часто скрывается за случайными колебаниями.')
print('Увеличение выборки делает оценку стабильнее; одна победа B сама по себе не доказывает статистическую значимость.')

7. Одинаковая сумма: 101,909,360 ₸.
Цикл: 601.11 мкс; NumPy: 9.18 мкс; ускорение 65.52 раза.
Ускорение измерено на этом компьютере; на другом устройстве результат изменится.

8. Один эксперимент: A — 54 покупок (5.4%); B — 58 (5.8%).
100 опытов: B лучше A в 84 случаях, A лучше B в 15, ничья в 1.
При 1000 клиентах на группу разница в 1 процентный пункт часто скрывается за случайными колебаниями.
Увеличение выборки делает оценку стабильнее; одна победа B сама по себе не доказывает статистическую значимость.
